In [ ]:
import numpy as np
import pandas as pd
import json
import re
import time
import os
import threading
import concurrent.futures
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass
from abc import ABC, abstractmethod
import networkx as nx
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import logging

from camel.models import ModelFactory
from camel.types import ModelPlatformType
from camel.agents import ChatAgent
from camel.messages import BaseMessage
from camel.toolkits import SearchToolkit

import os
import requests
from typing import List
from urllib.parse import urlparse

# Set up logging (output to both console and file)
import logging
from datetime import datetime

# Clear existing log handlers (prevent duplicate configuration)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Create log directory
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

# Generate log file name (including timestamp)
log_file = log_dir / f"cp_agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure log format
log_format = '%(asctime)s - %(levelname)s - %(message)s'

# Create formatter
formatter = logging.Formatter(log_format)

# Create console handler
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

# Create file handler
file_handler = logging.FileHandler(log_file, encoding='utf-8')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)

# Configure root logger
logging.root.setLevel(logging.INFO)
logging.root.addHandler(console_handler)
logging.root.addHandler(file_handler)

# Get module-specific logger
logger = logging.getLogger(__name__)

# Enable silent mode by default, turning off verbose logs from third-party libraries
third_party_loggers = [
    'camel', 'httpx', 'urllib3', 'openai', 'requests',
    'camel.agents', 'camel.models', 'camel.messages', 'camel.types'
]

for logger_name in third_party_loggers:
    third_party_logger = logging.getLogger(logger_name)
    third_party_logger.setLevel(logging.ERROR)  # Only show errors
    third_party_logger.propagate = False

# Test logging functionality
logger.info(f"Logging system initialized successfully (silent mode)")
logger.info(f"Log file saved at: {log_file}")
print(f"Logging configured, file saved at: {log_file}")
print(f"Silent mode enabled - Agent verbose logs turned off")


In [ ]:
# ================================
# Data structure definitions
# ================================

@dataclass
class PaperInfo:
    """Basic paper information"""
    paper_id: str
    title: str
    abstract: str
    authors: List[Dict[str, str]]  # List of dicts containing id, name, org
    venue: str
    year: int
    keywords: List[str]
    n_citation: int
    doc_type: str
    # Add missing fields
    references: List[str] = None
    page_start: str = ""
    page_end: str = ""
    lang: str = "en"
    volume: str = ""
    issue: str = ""
    issn: str = ""
    isbn: str = ""
    doi: str = ""
    url: List[str] = None
    # Add per-year citation fields
    citations_year_1: int = 0
    citations_year_2: int = 0
    citations_year_3: int = 0
    citations_year_4: int = 0
    citations_year_5: int = 0

@dataclass
class FeatureOutput:
    """Agent feature output"""
    agent_name: str
    features: Dict[str, float]
    evidence: Dict[str, str]
    confidence: float

print("Data structure definitions complete")


In [ ]:
class GitHubExplorer:
    """GitHub repository file browser"""

    def __init__(self, token: str):
        self.token = token
        self.headers = {
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github.v3+json"
        }

    def get_all_files(self, repo_url: str) -> List[str]:
        """Get all file paths in the repository"""
        repo_name = self._parse_repo_name(repo_url)
        print(f"Fetching the file list for repository {repo_name}...")

        files = self._get_files_recursive(repo_name, "")
        print(f"Successfully fetched {len(files)} files")
        return files

    def _get_files_recursive(self, repo_name: str, path: str) -> List[str]:
        """Recursively get the file list"""
        files = []

        try:
            url = f"https://api.github.com/repos/{repo_name}/contents/{path}"
            response = requests.get(url, headers=self.headers)

            if response.status_code == 200:
                contents = response.json()

                for item in contents:
                    if item['type'] == 'file':
                        files.append(item['path'])
                    elif item['type'] == 'dir':
                        sub_files = self._get_files_recursive(repo_name, item['path'])
                        files.extend(sub_files)
            else:
                print(f"API request failed: {response.status_code}")
                if response.status_code == 404:
                    print("Repository does not exist or no access permission")
                elif response.status_code == 403:
                    print("Access restricted, may require a token or rate limit reached")

        except Exception as e:
            print(f"Request failed: {e}")

        return files

    def _parse_repo_name(self, repo_url: str) -> str:
        """Parse the repository name, handling URL parameters"""
        # Clean the URL, removing query parameters and anchors
        if '?' in repo_url:
            repo_url = repo_url.split('?')[0]
        if '#' in repo_url:
            repo_url = repo_url.split('#')[0]

        if repo_url.startswith('https://github.com/'):
            return repo_url.replace('https://github.com/', '').rstrip('/')
        elif repo_url.startswith('git@github.com:'):
            return repo_url.replace('git@github.com:', '').replace('.git', '')
        else:
            return repo_url

print("GitHub repository file browser class definition complete")


In [ ]:
def explore_repo(token: str, repo_url: str):
    """Explore a GitHub repository"""
    explorer = GitHubExplorer(token)
    files = explorer.get_all_files(repo_url)

    return files


In [ ]:
# ================================
# Institution information retrieval utility class
# ================================

class InstitutionInfoRetriever:
    """Institution information retrieval utility class"""

    def __init__(self, debug_mode: bool = False):
        self.search_toolkit = SearchToolkit()
        self.institution_cache = {}  # Cache of already-retrieved institution information
        self.cache_lock = threading.Lock()  # Thread-safety lock
        self.debug_mode = debug_mode  # Debug mode, controls log output

    def get_institution_info(self, institution_name: str) -> str:
        """Get institution information, with caching support"""
        if not institution_name or institution_name.strip() == "" or institution_name == "Unknown":
            return ""

        # Normalize the institution name
        institution_name = institution_name.strip()

        # Check the cache
        with self.cache_lock:
            if institution_name in self.institution_cache:
                return self.institution_cache[institution_name]

        try:
            # Use SearchToolkit to retrieve Wikipedia information
            wiki_info = self.search_toolkit.search_wiki(institution_name)

            if wiki_info:
                # Extract key information (first 200 characters)
                summary = wiki_info[:200] + "..." if len(wiki_info) > 200 else wiki_info

                # Cache the result
                with self.cache_lock:
                    self.institution_cache[institution_name] = summary

                # Only output success log in debug mode
                if self.debug_mode:
                    logger.info(f"Successfully retrieved institution information: {institution_name}")
                return summary
            else:
                # Cache the empty result to avoid repeated retrieval
                with self.cache_lock:
                    self.institution_cache[institution_name] = ""
                return ""

        except Exception as e:
            # Only output warning in debug mode
            if self.debug_mode:
                logger.warning(f"Failed to retrieve institution information {institution_name}: {e}")
            # Cache the empty result
            with self.cache_lock:
                self.institution_cache[institution_name] = ""
            return ""

    def get_cache_stats(self) -> Dict[str, int]:
        """Get cache statistics"""
        with self.cache_lock:
            total = len(self.institution_cache)
            non_empty = sum(1 for info in self.institution_cache.values() if info)
            return {
                "total_cached": total,
                "non_empty_cached": non_empty,
                "empty_cached": total - non_empty
            }

    def set_debug_mode(self, debug_mode: bool):
        """Set debug mode"""
        self.debug_mode = debug_mode

print("Institution information retrieval utility class definition complete")


In [ ]:
# ================================
# Base Agent abstract class
# ================================

class BaseFeatureAgent(ABC):
    """Base class for feature-extraction Agents"""

    def __init__(self, model, agent_name: str):
        self.agent_name = agent_name
        self.model = model
        self.chat_agent = ChatAgent(
            system_message=self._get_system_prompt(),
            model=model,
            message_window_size=2
        )

    @abstractmethod
    def _get_system_prompt(self) -> str:
        """Get the system prompt"""
        pass

    @abstractmethod
    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        """Extract features"""
        pass

    def _validate_output(self, output: FeatureOutput) -> bool:
        """Validate the verifiability of the output"""
        return all([
            output.features,
            output.evidence,
            0 <= output.confidence <= 1
        ])

print("Base Agent abstract class definition complete")


In [ ]:
# ================================
# Concrete Agent implementation - Author Reputation Agent
# ================================

class AuthorReputationAgent(BaseFeatureAgent):
    """Author reputation analysis Agent"""

    def __init__(self, model, author_stats: Dict[str, Dict], institution_retriever: InstitutionInfoRetriever = None):
        super().__init__(model, "Author Reputation Agent")
        self.author_stats = author_stats
        self.institution_retriever = institution_retriever or InstitutionInfoRetriever()

    def _get_system_prompt(self) -> str:
        return """You are an author reputation assessment expert. Your task is to:
1. Assess the academic reputation of the authors
2. Base it on factors such as number of published papers, citation count, and institutional prestige
3. Give a score from 1 to 5, where 5 is the highest reputation

Output only the score, do not provide any explanation.
"""

    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        authors = paper.authors
        if not authors:
            return FeatureOutput(
                agent_name=self.agent_name,
                features={'author_reputation': 1.0},
                evidence={'error': 'No author information'},
                confidence=0.1
            )

        author_count = len(authors)

        # First author
        first_author = authors[0]['id']
        first_author_stats = self.author_stats.get(first_author, {'paper_count': 0, 'citation_count': 0, 'org': 'Unknown'})

        # Get the first author's institution information
        org1_info = self.institution_retriever.get_institution_info(first_author_stats.get('org', 'Unknown'))
        if 'no page' in org1_info:
            list_org1_info = org1_info.split(',')
            org1_info = self.institution_retriever.get_institution_info(list_org1_info[0])
            if 'no page' in org1_info:
                org1_info = 'Wikipedia does not have information about this institution'

        # Build the first author's analysis prompt
        prompt1 = f"""
        Assess the academic reputation of the following author:
        Author: {first_author}
        Institution: {first_author_stats.get('org', 'Unknown')}
        Number of published papers: {first_author_stats.get('paper_count', 0)}
        Total citations: {first_author_stats.get('citation_count', 0)}
        """

        if org1_info:
            prompt1 += f"Institution info: {org1_info}\n"

        prompt1 += "Please assess the overall author reputation (1-5). Output only the score, do not provide any explanation."

        response1 = self.chat_agent.step(prompt1)

        features = {
            'author1_reputation': response1.msg.content[:200]
        }

        evidence = {
            'author1_stats': f"{first_author}: {first_author_stats.get('paper_count', 0)} papers, {first_author_stats.get('citation_count', 0)} citations"
        }

        # Handle the last author and other authors based on the number of authors
        if author_count == 1:
            # Only one author; the last author's reputation equals the first author's
            features['author2_reputation'] = features['author1_reputation']
            features['other_authors_reputation'] = features['author1_reputation']
            evidence['author2_stats'] = evidence['author1_stats']
            evidence['other_authors_stats'] = "Same as the first author"

        elif author_count == 2:
            # Two authors; the last author is assessed separately, and other authors' reputation is the average of the two
            last_author = authors[1]['id']
            last_author_stats = self.author_stats.get(last_author, {'paper_count': 0, 'citation_count': 0, 'org': 'Unknown'})

            # Get the last author's institution information
            org2_info = self.institution_retriever.get_institution_info(last_author_stats.get('org', 'Unknown'))
            if 'no page' in org2_info:
                list_org2_info = org2_info.split(',')
                org2_info = self.institution_retriever.get_institution_info(list_org2_info[0])
                if 'no page' in org2_info:
                    org2_info = 'Wikipedia does not have information about this institution'

            prompt2 = f"""
            Assess the academic reputation of the following author:
            Author: {last_author}
            Institution: {last_author_stats.get('org', 'Unknown')}
            Number of published papers: {last_author_stats.get('paper_count', 0)}
            Total citations: {last_author_stats.get('citation_count', 0)}
            """

            if org2_info:
                prompt2 += f"Institution info: {org2_info}\n"

            prompt2 += "Please assess the overall author reputation (1-5). Output only the score, do not provide any explanation."

            response2 = self.chat_agent.step(prompt2)

            features['author2_reputation'] = response2.msg.content[:200]
            # Compute other authors' reputation (average of the first and last author)
            try:
                author1_score = float(features['author1_reputation'])
                author2_score = float(features['author2_reputation'])
                features['other_authors_reputation'] = str((author1_score + author2_score) / 2)
            except (ValueError, TypeError):
                features['other_authors_reputation'] = features['author1_reputation']
            evidence['author2_stats'] = f"{last_author}: {last_author_stats.get('paper_count', 0)} papers, {last_author_stats.get('citation_count', 0)} citations"
            evidence['other_authors_stats'] = "Average of the first and last author"

        else:
            # Three or more authors; assess the last author and other authors separately
            last_author = authors[-1]['id']
            last_author_stats = self.author_stats.get(last_author, {'paper_count': 0, 'citation_count': 0, 'org': 'Unknown'})

            # Get the last author's institution information
            org2_info = self.institution_retriever.get_institution_info(last_author_stats.get('org', 'Unknown'))
            if 'no page' in org2_info:
                list_org2_info = org2_info.split(',')
                org2_info = self.institution_retriever.get_institution_info(list_org2_info[0])
                if 'no page' in org2_info:
                    org2_info = 'Wikipedia does not have information about this institution'

            prompt2 = f"""
            Assess the academic reputation of the following author:
            Author: {last_author}
            Institution: {last_author_stats.get('org', 'Unknown')}
            Number of published papers: {last_author_stats.get('paper_count', 0)}
            Total citations: {last_author_stats.get('citation_count', 0)}
            """

            if org2_info:
                prompt2 += f"Institution info: {org2_info}\n"

            prompt2 += "Please assess the overall author reputation (1-5). Output only the score, do not provide any explanation."

            response2 = self.chat_agent.step(prompt2)

            # Handle other authors (middle authors)
            other_authors = authors[1:-1]  # Authors other than the first and last author
            other_authors_info = []

            for author in other_authors:
                author_id = author['id']
                author_stat = self.author_stats.get(author_id, {'paper_count': 0, 'citation_count': 0, 'org': 'Unknown'})
                other_authors_info.append({
                    'id': author_id,
                    'org': author_stat.get('org', 'Unknown'),
                    'paper_count': author_stat.get('paper_count', 0),
                    'citation_count': author_stat.get('citation_count', 0)
                })

            # Build the overall assessment prompt for other authors
            prompt_other = """
            Assess the overall academic reputation of the following group of authors:
            """

            for i, author_info in enumerate(other_authors_info, 1):
                prompt_other += f"""
                Author {i}: {author_info['id']}
                Institution: {author_info['org']}
                Number of published papers: {author_info['paper_count']}
                Total citations: {author_info['citation_count']}
                """

            prompt_other += "Please assess the overall academic reputation of these authors (1-5). Output only the score, do not provide any explanation."

            response_other = self.chat_agent.step(prompt_other)

            features['author2_reputation'] = response2.msg.content[:200]
            features['other_authors_reputation'] = response_other.msg.content[:200]
            evidence['author2_stats'] = f"{last_author}: {last_author_stats.get('paper_count', 0)} papers, {last_author_stats.get('citation_count', 0)} citations"
            evidence['other_authors_stats'] = f"Statistics for the other {len(other_authors)} authors"

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.85
        )

print("Author Reputation Agent definition complete")


In [ ]:
# ================================
# Venue Quality Agent
# ================================

class VenueQualityAgent(BaseFeatureAgent):
    """Venue quality Agent - supports external rating documents"""

    def __init__(self, model, ccf_file: str = None, core_file: str = None):
        super().__init__(model, "Venue Quality Agent")
        self.ccf_ratings = {}
        self.core_ratings = {}

        # Read the CCF rating document
        if ccf_file and os.path.exists(ccf_file):
            self._load_ratings(ccf_file, self.ccf_ratings)

        # Read the CORE rating document
        if core_file and os.path.exists(core_file):
            self._load_ratings(core_file, self.core_ratings)

    def _load_ratings(self, file_path: str, ratings_dict: Dict[str, str]):
        """Load rating data into the dictionary"""
        try:
            import csv

            with open(file_path, 'r', encoding='utf-8') as f:
                # Detect the file format
                first_line = f.readline().strip()
                f.seek(0)  # Reset the file pointer

                if first_line.startswith('conference,') or first_line.startswith('full_name,'):
                    # CSV format
                    reader = csv.DictReader(f)
                    for row in reader:
                        if 'conference' in row and 'rank' in row:
                            # CCF format: conference,full name,rank
                            conference = row['conference'].strip()
                            full_name = row.get('full name', '').strip()
                            rank = row['rank'].strip()

                            if conference and rank:
                                # Storage format: "conference,full_name,rank"
                                if full_name:
                                    ratings_dict[conference.lower()] = f"{conference},{full_name},{rank}"
                                else:
                                    ratings_dict[conference.lower()] = f"{conference},,{rank}"
                            if full_name and rank:
                                # Also store the full-name mapping
                                if conference:
                                    ratings_dict[full_name.lower()] = f"{conference},{full_name},{rank}"
                                else:
                                    ratings_dict[full_name.lower()] = f",{full_name},{rank}"

                        elif 'conference' in row and 'rank' in row:
                            # CORE format: full_name,conference,source,rank
                            conference = row['conference'].strip()
                            full_name = row['full_name'].strip()
                            rank = row['rank'].strip()

                            if conference and rank:
                                # Storage format: "conference,full_name,rank"
                                if full_name:
                                    ratings_dict[conference.lower()] = f"{conference},{full_name},{rank}"
                                else:
                                    ratings_dict[conference.lower()] = f"{conference},,{rank}"
                            if full_name and rank:
                                # Also store the full-name mapping
                                if conference:
                                    ratings_dict[full_name.lower()] = f"{conference},{full_name},{rank}"
                                else:
                                    ratings_dict[full_name.lower()] = f",{full_name},{rank}"
                else:
                    # The original txt format
                    for line in f:
                        line = line.strip()
                        if not line or line.startswith('#'):
                            continue

                        # Support multiple formats: venue_name,rating or venue_name:rating
                        if ',' in line:
                            parts = line.split(',', 1)
                        elif ':' in line:
                            parts = line.split(':', 1)
                        else:
                            continue

                        if len(parts) == 2:
                            venue = parts[0].strip()
                            rating = parts[1].strip()
                            ratings_dict[venue.lower()] = rating

            logger.info(f"Loaded {len(ratings_dict)} ratings")
        except Exception as e:
            logger.error(f"Failed to load rating file: {e}")

    def _find_venue_rating(self, venue: str) -> str:
        """Look up rating information for the conference/journal"""
        if not venue:
            return ""

        venue_lower = venue.lower()
        rating_info = []

        # Look up the CCF rating
        if venue_lower in self.ccf_ratings:
            rating_data = self.ccf_ratings[venue_lower]
            rating_info.append(f"CCF: {self._format_rating_info(rating_data)}")
        else:
            # Try a stricter fuzzy match
            best_match = None
            best_score = 0

            for ccf_venue, rating_data in self.ccf_ratings.items():
                # Compute the match score
                score = self._calculate_match_score(venue_lower, ccf_venue)
                if score > best_score and score > 0.6:  # Only accept high-confidence matches
                    best_score = score
                    best_match = (ccf_venue, rating_data)

            if best_match:
                rating_info.append(f"CCF: {self._format_rating_info(best_match[1])}")

        # Look up the CORE rating
        if venue_lower in self.core_ratings:
            rating_data = self.core_ratings[venue_lower]
            rating_info.append(f"CORE: {self._format_rating_info(rating_data)}")
        else:
            # Try a stricter fuzzy match
            best_match = None
            best_score = 0

            for core_venue, rating_data in self.core_ratings.items():
                # Compute the match score
                score = self._calculate_match_score(venue_lower, core_venue)
                if score > best_score and score > 0.6:  # Only accept high-confidence matches
                    best_score = score
                    best_match = (core_venue, rating_data)

            if best_match:
                rating_info.append(f"CORE: {self._format_rating_info(best_match[1])}")

        return " | ".join(rating_info) if rating_info else ""

    def _calculate_match_score(self, venue: str, rating_venue: str) -> float:
        """Compute the match score, returning a value between 0 and 1"""
        if not venue or not rating_venue:
            return 0.0

        # Exact match
        if venue == rating_venue:
            return 1.0

        # Check whether venue is a substring of rating_venue
        if venue in rating_venue:
            # Compute the length ratio to avoid overly short matches
            length_ratio = len(venue) / len(rating_venue)
            if length_ratio > 0.3:  # venue accounts for at least 30% of rating_venue
                return length_ratio * 0.8  # Substring match scores at most 0.8

        # Check whether rating_venue is a substring of venue
        if rating_venue in venue:
            length_ratio = len(rating_venue) / len(venue)
            if length_ratio > 0.3:  # rating_venue accounts for at least 30% of venue
                return length_ratio * 0.7  # Reverse substring match scores at most 0.7

        # Check keyword matching
        venue_words = set(venue.split())
        rating_words = set(rating_venue.split())

        if venue_words and rating_words:
            common_words = venue_words.intersection(rating_words)
            if len(common_words) > 0:
                # Compute the proportion of common words
                word_ratio = len(common_words) / max(len(venue_words), len(rating_words))
                return word_ratio * 0.6  # Keyword match scores at most 0.6

        return 0.0

    def _format_rating_info(self, rating_data: str) -> str:
        """Format the rating information, returning only the conference abbreviation, full name, and rating"""
        if not rating_data:
            return ""

        # Check whether this is the new CSV format (conference,full_name,rank)
        if ',' in rating_data and rating_data.count(',') >= 2:
            parts = rating_data.split(',', 2)
            conference = parts[0].strip()
            full_name = parts[1].strip()
            rank = parts[2].strip()

            # Build the concise format
            if conference and full_name:
                return f"{conference} ({full_name}): {rank}"
            elif conference:
                return f"{conference}: {rank}"
            elif full_name:
                return f"{full_name}: {rank}"
            else:
                return rank
        else:
            # The original simple format, return it directly
            return rating_data

    def _get_system_prompt(self) -> str:
        return """You are an academic venue expert. Assess:
1. The tier and influence of the conference/journal
2. Its reputation and recognition within the academic community
3. Give a score from 1 to 5, where 5 is a top-tier conference/journal

Output only the score, do not provide any explanation.
"""

    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        # Look up rating information for this conference/journal
        rating_info = self._find_venue_rating(paper.venue)

        prompt = f"""
        Assess venue quality:

        Conference/Journal: {paper.venue}
        Document type: {paper.doc_type}
        Publication year: {paper.year}
        """

        # Only add to the prompt when rating information is found
        if rating_info:
            prompt += f"\nOfficial rating: {rating_info}"

        prompt += "\n\nPlease assess the academic influence of this venue (1-5). Output only the score, do not provide any explanation."

        response = self.chat_agent.step(prompt)

        features = {
            'venue_score': response.msg.content[:200]
        }

        evidence = {
            'venue_classification': f"{paper.venue} - score: {response.msg.content[:200]}",
            'doc_type': paper.doc_type,
        }

        # If rating information exists, add it to the evidence
        if rating_info:
            evidence['official_ratings'] = rating_info

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.9 if rating_info else 0.7
        )

print("Venue Quality Agent definition complete")


In [ ]:
# ================================
# Other Agent implementations
# ================================

class ReproducibilityAgent(BaseFeatureAgent):
    """Reproducibility analysis Agent"""

    def __init__(self, model):
        super().__init__(model, "Reproducibility Analysis Agent")

    def _get_system_prompt(self) -> str:
        return """You are a reproducibility assessment expert. Analyze the reproducibility of the paper:
        Check whether the abstract mentions code, data, GitHub links, etc.
        """

    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        open_source_keywords = [
            'github', 'gitlab', 'bitbucket', 'Gitee', 'SourceForge', 'Codeberg', 'SourceHut', 'Launchpad', 'GNU Savannah', 'Kaggle'
        ]

        prompt1 = f"""
        Assess the reproducibility of the paper:

        Title: {paper.title}
        Abstract: {paper.abstract}

        Based on the following keywords, determine whether open-source code or data is provided:
        Keywords: {', '.join(open_source_keywords)}

        If any related keyword or any other open-source code or data link is found in the title or abstract, give a score of 1; otherwise give a score of 0.
        Output only the score, do not provide any explanation.
        """

        # Use the traditional approach to match GitHub links
        import re

        response1 = self.chat_agent.step(prompt1)

        # Search for GitHub links in the title and abstract
        text_to_search = f"{paper.title} {paper.abstract}"

        # Regex patterns for GitHub links
        github_patterns = [
            r'github\.com/[\w\-\.]+/[\w\-\.]+',
            r'www\.github\.com/[\w\-\.]+/[\w\-\.]+',
            r'https?://github\.com/[\w\-\.]+/[\w\-\.]+',
            r'https?://www\.github\.com/[\w\-\.]+/[\w\-\.]+'
        ]

        github_found = False
        repo_url = None
        for pattern in github_patterns:
            match = re.search(pattern, text_to_search)
            if match:
                github_found = True
                repo_url = match.group(0)
                if not repo_url.startswith('http'):
                    repo_url = 'https://' + repo_url
                break

        # If a GitHub link is found, use explore_repo to count the number of files
        files = []
        if github_found and repo_url:
            token = os.environ.get("GITHUB_TOKEN", "")
            files = explore_repo(token, repo_url[0:-1])
            print(len(files))
            if len(files) == 0:
                files = explore_repo(token, repo_url)
                print(f"Second attempt for GitHub link: {repo_url}")
            print(f"Number of files found: {len(files)}")
            print(f"GitHub link: {repo_url[0:-1]}")

        # Compute the final reproducibility score
        try:
            base_score = int(response1.msg.content[:200])
            if len(files) > 1:
                final_score = base_score + 1
            else:
                final_score = base_score + 0
        except (ValueError, TypeError):
            final_score = 0

        features = {
            'reproducibility': str(final_score)
        }

        evidence = {
            'llm_analysis': str(final_score)
        }

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.9
        )

class CollaborationNetworkAgent(BaseFeatureAgent):
    """Collaboration network analysis Agent"""

    def __init__(self, model, institution_retriever: InstitutionInfoRetriever = None):
        super().__init__(model, "Collaboration Network Analysis Agent")
        self.institution_retriever = institution_retriever or InstitutionInfoRetriever()

    def _get_system_prompt(self) -> str:
        return """You are an academic collaboration network expert. Analyze:
1. Whether cross-national collaboration exists
2. The team size
3. The reputation and diversity of the institutions
4. Give an overall score from 1 to 5

Output only the score, do not provide any explanation.
"""

    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        author_info = []
        institution_details = []

        for author in paper.authors:
            org = author.get('org', 'Unknown')
            author_info.append(f"{author['name']} ({org})")

            # Get detailed institution information
            if org and org != 'Unknown':
                org_info = self.institution_retriever.get_institution_info(org)
                if 'no page' in org_info:
                    list_org_info = org_info.split(',')
                    org_info = self.institution_retriever.get_institution_info(list_org_info[0])
                    if 'no page' in org_info:
                        org_info = 'Wikipedia does not have information about this institution'
                if org_info:
                    institution_details.append(f"{org}: {org_info}")

        prompt = f"""
        Analyze the academic collaboration network:

        Author information: {'; '.join(author_info)}
        """
        institution_details = list(set(institution_details))
        if institution_details:
            prompt += f"\nInstitution details:\n" + "\n".join(institution_details)

        prompt += "\nPlease assess the quality and diversity of the collaboration network (1-5). Output only a single overall score, do not provide any explanation."

        response = self.chat_agent.step(prompt)

        features = {
            'collaboration_network': response.msg.content[:200]
        }

        evidence = {
            'team_composition': f"Team size: {len(paper.authors)} people",
            'institutions': self._get_institution_info(paper.authors),
            'llm_analysis': response.msg.content[:200]
        }

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.75
        )

    def _get_institution_info(self, authors: List[Dict[str, str]]) -> str:
        """Get a summary of institution information"""
        institutions = []
        for author in authors:
            org = author.get('org', 'Unknown')
            if org not in institutions:
                institutions.append(org)

        return f"Institutions involved: {len(institutions)}"

class FieldHotnessAgent(BaseFeatureAgent):
    """Field hotness analysis Agent"""

    def __init__(self, model, keyword_stats: Dict[str, Dict]):
        super().__init__(model, "Field Hotness Agent")
        self.keyword_stats = keyword_stats

    def _get_system_prompt(self) -> str:
        return """You are an academic trend analysis expert. Analyze:
1. The current popularity of the research field
2. The popularity trend of the keywords
3. Give a hotness score from 1 to 5

Output only the score, do not provide any explanation.
"""

    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        hotness_score = self._calculate_field_hotness(paper.keywords, paper.year)

        features = {
            'field_hotness': hotness_score
        }

        evidence = {
            'keyword_analysis': self._get_keyword_evidence(paper.keywords, paper.year),
            'hotness_calculation': f"Computed from the annual publication counts of {len(paper.keywords)} keywords",
        }

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.8
        )

    def _calculate_field_hotness(self, keywords: List[str], paper_year: int) -> float:
        """Compute field hotness"""
        if not keywords:
            return 1.0

        target_year = paper_year - 1 if paper_year > 2010 else paper_year

        keyword_counts = []
        for keyword in keywords:
            keyword_lower = keyword.lower()
            stats = self.keyword_stats.get(keyword_lower, {})
            count = stats.get(target_year, 0)
            keyword_counts.append(count)

        if not keyword_counts:
            return 1.0

        return np.mean(keyword_counts)

    def _get_keyword_evidence(self, keywords: List[str], paper_year: int) -> str:
        """Get keyword hotness evidence"""
        target_year = paper_year - 1 if paper_year > 2010 else paper_year

        evidence_parts = []
        for keyword in keywords[:3]:
            keyword_lower = keyword.lower()
            stats = self.keyword_stats.get(keyword_lower, {})
            count = stats.get(target_year, 0)
            evidence_parts.append(f"{keyword}: {count} papers")

        return f"Publication count in {target_year}: " + ", ".join(evidence_parts)

class TitleQualityAgent(BaseFeatureAgent):
    """Title quality analysis Agent"""

    def __init__(self, model):
        super().__init__(model, "Title Quality Analysis Agent")

    def _get_system_prompt(self) -> str:
        return """You are a title quality assessment expert. Analyze:
1. The clarity and accuracy of the title
2. The appeal and professionalism of the title
3. Give a quality score from 1 to 5

Refer to the following high-quality title examples:

Shot1:
Title: Train for the Worst, Plan for the Best: Understanding Token Ordering in Masked Diffusions

Shot2:
Title: Learning Dynamics of LLM Finetuning

Shot3
Title: Not All Tokens Are What You Need for Pretraining

Output only the score, do not provide any explanation. The score may include decimals.
"""

    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        prompt = f"""
        Please assess the quality of the title: {paper.title} (1-5). Score strictly to ensure clear differentiation between different titles; output only the score. Do not provide any explanation. The score may include decimals.
        """

        response = self.chat_agent.step(prompt)

        features = {
            'title_quality': response.msg.content[:200]
        }

        evidence = {
            'llm_assessment': response.msg.content[:200]
        }

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.85
        )

class ContentQualityAgent(BaseFeatureAgent):
    """Abstract quality analysis Agent"""

    def __init__(self, model):
        super().__init__(model, "Content Quality Analysis Agent")

    def _get_system_prompt(self) -> str:
        return """You are an abstract quality assessment expert. Analyze:
1. The structure and completeness of the abstract
2. The professional level of the content expression
3. Give a quality score from 1 to 5

Output only the score, do not provide any explanation. The score may include decimals.
"""
# Refer to the following high-quality abstract examples; do not score these abstracts:
# Shot1:
# Abstract: In recent years, masked diffusion models (MDMs) have emerged as a promising alternative approach for generative modeling over discrete domains. Compared to autoregressive models (ARMs), MDMs trade off complexity at training time with flexibility at inference time. At training time, they must learn to solve an exponentially large number of infilling problems, but at inference time, they can decode tokens in essentially arbitrary order. In this work we closely examine these two competing effects. On the training front, we theoretically and empirically demonstrate that MDMs indeed train on computationally intractable subproblems compared to their autoregressive counterparts. On the inference front, we show that a suitable strategy for adaptively choosing the token decoding order significantly enhances the capabilities of MDMs, allowing them to sidestep hard subproblems. On logic puzzles like Sudoku, we show that adaptive inference can boost solving accuracy in pretrained MDMs from <7% to ≈90%, even outperforming ARMs that were explicitly trained via teacher forcing to learn the right order of decoding.

# Shot2:
# Abstract: Learning dynamics, which describes how the learning of specific training examples influences the model's predictions on other examples, gives us a powerful tool for understanding the behavior of deep learning systems. We study the learning dynamics of large language models during different types of finetuning, by analyzing the step-wise decomposition of how influence accumulates among different potential responses. Our framework allows a uniform interpretation of many interesting observations about the training of popular algorithms for both instruction tuning and preference tuning. In particular, we propose a hypothetical explanation of why specific types of hallucination are strengthened after finetuning, e.g., the model might use phrases or facts in the response for question B to answer question A, or the model might keep repeating similar simple phrases when generating responses. We also extend our framework and highlight a unique ``squeezing effect'' to explain a previously observed phenomenon in off-policy direct preference optimization (DPO), where running DPO for too long makes even the desired outputs less likely. This framework also provides insights into where the benefits of on-policy DPO and other variants come from. The analysis not only provides a novel perspective of understanding LLM's finetuning but also inspires a simple, effective method to improve alignment performance.

# Shot3
# Abstract: Previous language model pre-training methods have uniformly applied a next-token prediction loss to all training tokens. Challenging this norm, we posit that ''Not all tokens in a corpus are equally important for language model training''. Our initial analysis examines token-level training dynamics of language model, revealing distinct loss patterns for different tokens. Leveraging these insights, we introduce a new language model called Rho-1. Unlike traditional LMs that learn to predict every next token in a corpus, Rho-1 employs Selective Language Modeling (SLM), which selectively trains on useful tokens that aligned with the desired distribution. This approach involves scoring training tokens using a reference model, and then training the language model with a focused loss on tokens with higher scores. When continual continual pretraining on 15B OpenWebMath corpus, Rho-1 yields an absolute improvement in few-shot accuracy of up to 30% in 9 math tasks. After fine-tuning, Rho-1-1B and 7B achieved state-of-the-art results of 40.6% and 51.8% on MATH dataset, respectively - matching DeepSeekMath with only 3% of the pretraining tokens. Furthermore, when continual pretraining on 80B general tokens, Rho-1 achieves 6.8% average enhancement across 15 diverse tasks, increasing both data efficiency and performance of the language model pre-training.


    def extract_features(self, paper: PaperInfo, context: Dict[str, Any]) -> FeatureOutput:
        prompt = f"""
        Please assess the quality of the abstract: {paper.abstract} (1-5). Score strictly to ensure clear differentiation between different abstracts; output only the score, do not provide any explanation.
        """

        response = self.chat_agent.step(prompt)

        features = {
            'content_quality': response.msg.content[:200]
        }

        evidence = {
            'llm_assessment': response.msg.content[:200]
        }

        return FeatureOutput(
            agent_name=self.agent_name,
            features=features,
            evidence=evidence,
            confidence=0.85
        )

print("Other Agent class definitions complete")


In [ ]:
# ================================
# Batch writer (thread-safe)
# ================================

class ThreadSafeBatchWriter:
    """Thread-safe batch writer"""

    def __init__(self, output_file: str, batch_size: int = 100):
        self.output_file = output_file
        self.batch_size = batch_size
        self.buffer = []
        self.written_count = 0
        self.lock = threading.Lock()

        # Initialize the output file
        with open(self.output_file, 'w', encoding='utf-8') as f:
            pass

        logger.info(f"Batch writer initialized: {output_file}, batch size: {batch_size}")

    def add_result(self, result: Dict[str, Any]):
        """Add a result to the buffer"""
        with self.lock:
            self.buffer.append(result)

            # Write when the batch size is reached
            if len(self.buffer) >= self.batch_size:
                self._flush_buffer()

    def _flush_buffer(self):
        """Write the buffer to the file (must be called within the lock)"""
        if not self.buffer:
            return

        try:
            with open(self.output_file, 'a', encoding='utf-8') as f:
                for result in self.buffer:
                    f.write(json.dumps(result, ensure_ascii=False) + '\n')

            self.written_count += len(self.buffer)
            # Reduce log frequency, only output for larger batches
            if self.written_count % 500 == 0 or len(self.buffer) >= self.batch_size:
                logger.info(f"Batch write: +{len(self.buffer)} entries, total: {self.written_count} records")
            self.buffer = []

        except Exception as e:
            logger.error(f"Batch write failed: {e}")

    def finalize(self):
        """Finish writing, handling the remaining buffer"""
        with self.lock:
            if self.buffer:
                logger.info(f"Writing remaining {len(self.buffer)} results")
                self._flush_buffer()

        logger.info(f"Batch writing complete, {self.written_count} results written in total")

    def get_written_count(self) -> int:
        """Get the number of records written"""
        with self.lock:
            return self.written_count

print("Batch writer definition complete")


In [ ]:
# ================================
# Multithreaded coordinator
# ================================

class MultithreadedCoordinator:
    """Multithreaded feature-extraction coordinator"""

    def __init__(self, model, author_stats: Dict, keyword_stats: Dict, ccf_file: str = None, core_file: str = None):
        self.model = model
        self.author_stats = author_stats
        self.keyword_stats = keyword_stats
        self.ccf_file = ccf_file
        self.core_file = core_file
        self.institution_retriever = InstitutionInfoRetriever(debug_mode=False)  # Debug mode off by default

        # Set the global institution retriever for convenient external control of debug mode
        global institution_retriever_global
        institution_retriever_global = self.institution_retriever

    def create_agents(self):
        """Create independent Agent instances for each thread"""
        return {
            'author_reputation': AuthorReputationAgent(self.model, self.author_stats, self.institution_retriever),
            'venue_quality': VenueQualityAgent(self.model, self.ccf_file, self.core_file),
            'reproducibility': ReproducibilityAgent(self.model),
            'collaboration_network': CollaborationNetworkAgent(self.model, self.institution_retriever),
            'field_hotness': FieldHotnessAgent(self.model, self.keyword_stats),
            'title_quality': TitleQualityAgent(self.model),
            'content_quality': ContentQualityAgent(self.model)
        }

    def analyze_paper(self, paper: PaperInfo, agents: Dict, context: Dict[str, Any] = None) -> Dict[str, Any]:
        """Analyze a single paper"""
        if context is None:
            context = {}

        results = {}

        for agent_name, agent in agents.items():
            try:
                result = agent.extract_features(paper, context)
                if agent._validate_output(result):
                    results[agent_name] = result
            except Exception as e:
                logger.error(f"Agent {agent_name} failed to process paper {paper.paper_id}: {e}")

        # Generate the feature report
        feature_scores = {}
        for result in results.values():
            for feature_name, score in result.features.items():
                feature_scores[feature_name] = score

        feature_scores.update({
        'paper_id': paper.paper_id,
        'title': paper.title,
        'abstract': paper.abstract,
        'authors': paper.authors,
        'venue': paper.venue,
        'year': paper.year,
        'keywords': paper.keywords,
        'n_citation': paper.n_citation,
        'doc_type': paper.doc_type,
        'references': paper.references,
        'page_start': paper.page_start,
        'page_end': paper.page_end,
        'lang': paper.lang,
        'volume': paper.volume,
        'issue': paper.issue,
        'issn': paper.issn,
        'isbn': paper.isbn,
        'doi': paper.doi,
        'url': paper.url,
        # Per-year citation data
        'citations_year_1': paper.citations_year_1,
        'citations_year_2': paper.citations_year_2,
        'citations_year_3': paper.citations_year_3,
        'citations_year_4': paper.citations_year_4,
        'citations_year_5': paper.citations_year_5,
    })
        return {
            paper.paper_id: feature_scores
        }

def process_paper_chunk(papers_chunk: List[PaperInfo], thread_id: int,
                       coordinator: MultithreadedCoordinator, writer: ThreadSafeBatchWriter,
                       progress_interval: int = 10) -> int:
    """Worker function that processes a chunk of papers"""

    logger.info(f"Thread {thread_id}: starting to process {len(papers_chunk)} papers")

    # Create independent Agent instances for this thread
    agents = coordinator.create_agents()
    processed_count = 0

    # Use simple log output instead of a progress bar to avoid multithreading conflicts
    for i, paper in enumerate(papers_chunk):
        try:
            # Analyze the paper features
            result = coordinator.analyze_paper(paper, agents)

            # Write the result
            writer.add_result(result)
            processed_count += 1

            # Report progress periodically
            if (i + 1) % progress_interval == 0 or (i + 1) == len(papers_chunk):
                progress_percent = ((i + 1) / len(papers_chunk)) * 100
                logger.info(f"Thread {thread_id}: progress {progress_percent:.1f}% ({i+1}/{len(papers_chunk)}) - processed: {processed_count}, total written: {writer.get_written_count()}")

        except Exception as e:
            logger.error(f"Thread {thread_id} error while processing paper {paper.paper_id}: {e}")
            continue

    logger.info(f"Thread {thread_id}: finished, processed {processed_count} papers")
    return processed_count

print("Multithreaded coordinator definition complete")


In [ ]:
# ================================
# Data processing pipeline class
# ================================

class DataProcessor:
    """Data processor - responsible for processing the sampled data"""

    def __init__(self, ccf_file: str = None, core_file: str = None):
        # Initialize the CAMEL model
        self.llm_model = ModelFactory.create(
            model_platform=ModelPlatformType.OLLAMA,
            model_type="qwen2.5:7b",
            url="http://localhost:11434/v1",
            model_config_dict={"temperature": 0.0,"max_tokens": 1024}
        )

        self.ccf_file = ccf_file
        self.core_file = core_file

        # Initialize the coordinator (statistics will be set later)
        self.coordinator = None

    def load_sampled_data(self, data_path: str) -> List[PaperInfo]:
        """Load the sampled data"""
        papers = []
        print(f"Loading sampled data file: {data_path}")

        with open(data_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                try:
                    data = json.loads(line.strip())
                    paper = PaperInfo(
                        paper_id=data['id'],
                        title=data['title'],
                        abstract=data.get('abstract', ''),
                        authors=data.get('authors', []),
                        venue=data.get('venue', ''),
                        year=data.get('year', 2010),
                        keywords=data.get('keywords', []),
                        n_citation=data.get('n_citation', 0),
                        doc_type=data.get('doc_type', 'Unknown'),
                        # Add missing fields
                        references=data.get('references', []),
                        page_start=data.get('page_start', ''),
                        page_end=data.get('page_end', ''),
                        lang=data.get('lang', 'en'),
                        volume=data.get('volume', ''),
                        issue=data.get('issue', ''),
                        issn=data.get('issn', ''),
                        isbn=data.get('isbn', ''),
                        doi=data.get('doi', ''),
                        url=data.get('url', []),
                        # Add per-year citation fields
                        citations_year_1=data.get('citations_year_1', 0),
                        citations_year_2=data.get('citations_year_2', 0),
                        citations_year_3=data.get('citations_year_3', 0),
                        citations_year_4=data.get('citations_year_4', 0),
                        citations_year_5=data.get('citations_year_5', 0)
                    )
                    papers.append(paper)

                    if line_num % 1000 == 0 and line_num > 0:
                        print(f"Loaded {line_num + 1} papers")

                except json.JSONDecodeError as e:
                    print(f"Error parsing line {line_num + 1}: {e}")
                except Exception as e:
                    print(f"Error processing line {line_num + 1}: {e}")

        print(f"Sampled data loading complete, {len(papers)} papers in total")
        return papers

    def build_statistics(self, papers: List[PaperInfo]) -> Tuple[Dict, Dict]:
        """Build author and keyword statistics"""
        print("Building statistics...")

        author_stats = defaultdict(lambda: {'paper_count': 0, 'citation_count': 0, 'org': 'Unknown'})
        keyword_stats = defaultdict(lambda: defaultdict(int))

        for paper in papers:
            # Author statistics
            for author in paper.authors:
                author_id = author['id']
                author_stats[author_id]['paper_count'] += 1
                author_stats[author_id]['citation_count'] += paper.n_citation
                author_stats[author_id]['org'] = author.get('org', 'Unknown')

            # Keyword statistics
            year = paper.year
            for keyword in paper.keywords:
                keyword_stats[keyword.lower()][year] += 1

        print(f"Statistics built: {len(author_stats)} authors, {len(keyword_stats)} keywords")
        return dict(author_stats), dict(keyword_stats)

    def process_papers_multithreaded(self, papers: List[PaperInfo], output_file: str,
                               max_workers: int = None, batch_size: int = 100,
                               progress_interval: int = 10) -> int:
        """Process papers with multiple threads"""

        if max_workers is None:
            max_workers = min(4, os.cpu_count() - 1)  # Conservative thread count

        logger.info(f"Starting multithreaded processing with {max_workers} threads")
        logger.info(f"Number of papers: {len(papers)}, output: {output_file}")
        logger.info(f"Batch size: {batch_size}, progress interval: {progress_interval}")

        # Create the batch writer
        writer = ThreadSafeBatchWriter(output_file, batch_size)

        # Split the data among the threads
        chunk_size = len(papers) // max_workers
        remainder = len(papers) % max_workers

        paper_chunks = []
        start_idx = 0

        for i in range(max_workers):
            # Handle the remainder; the first few threads get one extra paper
            current_chunk_size = chunk_size + (1 if i < remainder else 0)
            end_idx = start_idx + current_chunk_size

            if start_idx >= len(papers):
                break

            chunk = papers[start_idx:end_idx]
            paper_chunks.append(chunk)

            logger.info(f"Thread {i}: assigned {len(chunk)} papers (indices {start_idx}-{end_idx-1})")
            start_idx = end_idx

        # Process using a thread pool
        start_time = time.time()
        total_processed = 0

        with concurrent.futures.ThreadPoolExecutor(max_workers=len(paper_chunks)) as executor:
            # Submit tasks
            future_to_thread = {}
            for i, chunk in enumerate(paper_chunks):
                future = executor.submit(process_paper_chunk, chunk, i,
                                       self.coordinator, writer, progress_interval)
                future_to_thread[future] = i

            # Wait for all tasks to complete (using clear log output)
            logger.info(f"Waiting for {len(future_to_thread)} threads to complete...")
            completed_threads = 0

            for future in concurrent.futures.as_completed(future_to_thread):
                thread_id = future_to_thread[future]
                try:
                    processed_count = future.result(timeout=3600)  # 1-hour timeout
                    total_processed += processed_count
                    completed_threads += 1
                    logger.info(f"✓ Thread {thread_id} done: processed {processed_count} papers ({completed_threads}/{len(future_to_thread)} threads complete)")
                    logger.info(f"  Current total progress: {total_processed} processed, {writer.get_written_count()} records written")
                except Exception as e:
                    completed_threads += 1
                    logger.error(f"✗ Thread {thread_id} failed: {e} ({completed_threads}/{len(future_to_thread)} threads complete)")

        # Finish writing
        writer.finalize()

        elapsed_time = time.time() - start_time
        logger.info(f"Multithreaded processing complete: {len(papers)} -> {total_processed} papers")
        logger.info(f"Processing time: {elapsed_time:.2f} seconds")
        logger.info(f"Output file: {output_file}")

        return total_processed

print("⚙️ Data processing pipeline class definition complete")


In [ ]:
# ================================
# Main execution functions
# ================================

def process_sampled_datasets(ccf_file: str = None, core_file: str = None):
    """Process the sampled datasets"""
    print("🚀 Starting to process the sampled datasets")
    print("=" * 60)

    # Configuration parameters
    data_dir = "maindata/test1"
    output_dir = "maindata/test1/qwen3"

    # Dataset file paths
    dataset_files = {
        'train': os.path.join(data_dir, "train.jsonl"),
        'eval': os.path.join(data_dir, "eval.jsonl"),
        'test': os.path.join(data_dir, "test.jsonl")
    }

    if ccf_file or core_file:
        print(f"Using rating files:")
        if ccf_file:
            print(f"  CCF rating: {ccf_file}")
        if core_file:
            print(f"  CORE rating: {core_file}")
    else:
        print("No rating file provided; the default assessment method will be used")

    try:
        # Initialize the data processor
        processor = DataProcessor(ccf_file, core_file)

        # Process each dataset
        for dataset_name, input_file in dataset_files.items():
            if not os.path.exists(input_file):
                print(f"⚠️ Skipping non-existent file: {input_file}")
                continue

            print(f"\n📊 Processing the {dataset_name} dataset...")
            print(f"Input file: {input_file}")

            # Load the sampled data
            papers = processor.load_sampled_data(input_file)

            if not papers:
                print(f"⚠️ The {dataset_name} dataset is empty; skipping processing")
                continue

            # Build statistics
            author_stats, keyword_stats = processor.build_statistics(papers)

            # Initialize the coordinator
            processor.coordinator = MultithreadedCoordinator(
                processor.llm_model, author_stats, keyword_stats, ccf_file, core_file
            )

            # Set the output file
            output_file = os.path.join(output_dir, f"{dataset_name}_processsed.jsonl")

            # Multithreaded processing
            processed_count = processor.process_papers_multithreaded(
                papers, output_file,
                max_workers=16,          # Use 16 threads
                batch_size=200,          # Batch size 200
                progress_interval=100    # Report progress every 100 papers
            )

            print(f"✅ The {dataset_name} dataset processing is complete: {processed_count} papers")
            print(f"📁 Saved to: {output_file}")

        print("\n🎉 All datasets processed!")

    except KeyboardInterrupt:
        print("\nUser interrupted processing")
        print("Already-processed data has been saved via batch writing")
    except Exception as e:
        print(f"\nProcessing failed: {str(e)}")
        logger.exception("Detailed error information:")

print("🎯 Main execution function definition complete")


In [ ]:
# ================================
# Agent log control utilities
# ================================

def disable_agent_logs():
    """Turn off the Agent's verbose log output"""
    import logging

    # Turn off CAMEL-related logs
    camel_loggers = [
        'camel',
        'camel.agents',
        'camel.models',
        'camel.messages',
        'camel.types',
        'httpx',
        'urllib3',
        'openai',
        'requests'
    ]

    for logger_name in camel_loggers:
        logger = logging.getLogger(logger_name)
        logger.setLevel(logging.ERROR)  # Only show errors
        logger.propagate = False

    print("🔇 Agent verbose logs turned off, keeping only error messages")

def enable_agent_logs():
    """Re-enable the Agent's verbose log output"""
    import logging

    camel_loggers = [
        'camel',
        'camel.agents',
        'camel.models',
        'camel.messages',
        'camel.types',
        'httpx',
        'urllib3',
        'openai',
        'requests'
    ]

    for logger_name in camel_loggers:
        logger = logging.getLogger(logger_name)
        logger.setLevel(logging.INFO)
        logger.propagate = True

    print("🔊 Agent verbose logs enabled")

def quiet_mode():
    """Silent mode - only show our program's key information"""
    import logging

    # Turn off third-party library logs
    third_party_loggers = [
        'camel', 'httpx', 'urllib3', 'openai', 'requests',
        'camel.agents', 'camel.models', 'camel.messages'
    ]

    for logger_name in third_party_loggers:
        logger = logging.getLogger(logger_name)
        logger.setLevel(logging.CRITICAL)  # Show almost no information
        logger.propagate = False

    # Keep our own logs at INFO level
    our_logger = logging.getLogger(__name__)
    our_logger.setLevel(logging.INFO)

    print("🤫 Silent mode enabled - only showing the program's core progress information")

def verbose_mode():
    """Verbose mode - show all debug information"""
    import logging

    # Enable all logs
    all_loggers = [
        'camel', 'httpx', 'urllib3', 'openai', 'requests',
        'camel.agents', 'camel.models', 'camel.messages'
    ]

    for logger_name in all_loggers:
        logger = logging.getLogger(logger_name)
        logger.setLevel(logging.DEBUG)
        logger.propagate = True

    print("🔍 Verbose mode enabled - showing all debug information")

def control_institution_debug_mode(debug_mode: bool = False):
    """Control the debug mode of institution information retrieval"""

    global institution_retriever_global

    # If the global institution retriever exists, set its debug mode
    if 'institution_retriever_global' in globals():
        institution_retriever_global.set_debug_mode(debug_mode)
        print(f"🏫 Institution information retrieval debug mode: {'enabled' if debug_mode else 'disabled'}")
    else:
        print("⚠️ The global institution retriever is not initialized; use this after creating MultithreadedCoordinator")

print("🎛️  Agent log control utilities are ready:")
print("  • disable_agent_logs() - turn off Agent verbose logs")
print("  • quiet_mode() - silent mode (recommended)")
print("  • enable_agent_logs() - enable Agent verbose logs")
print("  • verbose_mode() - verbose mode")
print("  • control_institution_debug_mode(True/False) - control institution information retrieval logs")
print("\n💡 Suggested run: quiet_mode() for clean output")
print("🏫 Institution information retrieval logs are off by default; for debugging run: control_institution_debug_mode(True)")


In [ ]:
# ================================
# Main execution code
# ================================

if __name__ == "__main__":
    # Configure rating file paths
    # Use your CSV-format rating files
    ccf_file = "ratings/ccf_data.csv"  # CCF rating CSV file
    core_file = "ratings/core_conference_rankings.csv"  # CORE rating CSV file

    # Enable silent mode
    quiet_mode()

    # Process the sampled datasets
    process_sampled_datasets(ccf_file, core_file)
